### MiddleWare

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
1) Tracking agent behavior with logging, analytics and debugging
2) Transforming prompts , tool selection , and output formatting
3) Adding retries, fallbacks and early termination logic
4) Applying rate limits, guardrails and PII detection

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


Built in middlewares -> Summarization, HumanInTheLoop,ModelCallLimit etc

### Summarisation MiddleWare
Automatically summarizes conversation history when approaching the token limits,preserving recent messages while compressing older context. Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conv context matters.

In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
#from langchain.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage

agent = create_agent(
    model="groq:qwen/qwen3-32b", 
    middleware=[
    SummarizationMiddleware(
        model="groq:qwen/qwen3-32b",
        trigger=("messages",5),
        keep=("messages",4))], 
    #checkpoint_saver=InMemorySaver()
    )




In [13]:
## Run with thread id
config={"configurable": {"thread_id": "test-1"}}

In [14]:
questions=[
    "What is the capital of France?",
    "What is the population of France?",
    "What is the GDP of France?",
    "What is the largest city in France?",
    "What is the official language of France?",
    "What is the currency of France?",
    "What is the time zone of France?",
    "What is the climate of France?",
    "What are some popular tourist attractions in France?",
    "What is the history of France?"
]

for q in questions:
    response = agent.invoke({"messages":[ HumanMessage(content=q)]},config)
    print(response)
    print(f"Messages in thread: {len(response['messages'])}")

{'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='104fc109-708a-42a3-acda-ab3e955b8bf1'), AIMessage(content='<think>\nOkay, so the user is asking for the capital of France. Let me think about this. I remember from school that France is a country in Europe. The capital... I think it\'s Paris. But wait, am I sure? Let me try to recall. Paris is known for the Eiffel Tower, the Louvre Museum, and it\'s a major city. I don\'t think it\'s Lyon or Marseille. Maybe Bordeaux? No, those are other cities in France. Let me double-check. I\'ve heard of the term "City of Light" referring to Paris, which is a nickname for the capital. Also, when people talk about French fashion or art, Paris is usually the example. So, yes, I\'m pretty confident it\'s Paris. I don\'t think there\'s any other city that\'s the capital. Maybe check if there\'s been any changes recently, but I don\'t think so. The capital hasn\'t changed in a long time. S

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
#from langchain.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool

@tool
def search_hotels(city:str)->str:
    """search hotels -returns long response to use more tokens"""
    return f"""Here are some hotels in {city}:
    1.Grand  Hotel - 5 stars, $350 per night
    2. City Inn - 3 stars, $150 per night       
    3. Budget Motel - 2 stars, $80 per night """
    
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_hotels],
    middleware=[SummarizationMiddleware(
        model="groq:qwen/qwen3-32b",
        trigger=("tokens",550),
        keep=("tokens",200)
    )]
)
config={"configurable": {"thread_id": "test-1"}}

def count_tokens(messages):
    total_chars=sum(len(m.content) for m in messages)
    return total_chars//4



In [16]:
cities=["Paris","New York","Tokyo","London","Sydney"]
for city in cities:
    response = agent.invoke({"messages":[ HumanMessage(content=f"Find me hotels in {city}")]},config)
    
    tokens=count_tokens(response["messages"])
    print(response["messages"])
    print(f"Tokens in thread: {tokens}, Messages in thread: {len(response['messages'])}")

[HumanMessage(content='Find me hotels in Paris', additional_kwargs={}, response_metadata={}, id='918180b3-b7e2-4a92-a24d-a87f52b759b4'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to find hotels in Paris. Let me check the available tools. There's a function called search_hotels that takes a city parameter. The city here is Paris. I need to make sure the function is called correctly. The parameters should be a JSON object with the city name. I'll format the tool call as specified, using the XML tags and proper JSON structure. Let me double-check the spelling of the city and the function name. Everything looks good. Time to output the tool call.\n", 'tool_calls': [{'id': 'sbhej3gf2', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 129, 'prompt_tokens': 156, 'total_tokens': 285, 'completion_time': 0.20836368, 'completion_tokens_details': {'reasoni

based on token size, messages, fraction

### Human In the Loop
Pause agent execution for human intervention


In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str)->str:
    """Reads an email and returns its content."""
    return f"Email content for {email_id}"

def send_email_tool(recipient:str, subject:str, body:str)->str:
    """Sends an email and returns a confirmation message."""
    return f"Email sent to {recipient} with subject '{subject}'"




In [21]:
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool":{"allowed_decisions":["approve","reject"],"read_email_tool":False}},
            #human_message=SystemMessage(content="You are an assistant that helps manage emails. You have access to two tools: read_email and send_email. Always use the tools when appropriate and ask the user for necessary information to use the tools effectively.")
        )
    ]
)

In [23]:
config={"configurable": {"thread_id": "test-1"}}

response = agent.invoke({"messages":[ HumanMessage(content="Send an email to John with the subject 'Meeting' and body 'Let's meet tomorrow at 10am.'")]},config)



In [24]:
response

{'messages': [HumanMessage(content="Send an email to John with the subject 'Meeting' and body 'Let's meet tomorrow at 10am.'", additional_kwargs={}, response_metadata={}, id='a1723787-757a-456b-acc4-6f0f019f1ec4'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to John. Let me check the available tools. There's the send_email_tool. The parameters required are recipient, subject, and body. The user provided all three: recipient is John, subject is 'Meeting', and body is 'Let's meet tomorrow at 10am.' So I need to call the send_email_tool with these arguments. I'll make sure the JSON structure is correct and includes all required fields. No need to use the read_email_tool here since the task is about sending, not reading. Double-checking the parameters to avoid missing any. Everything looks good. Time to format the tool call properly.\n", 'tool_calls': [{'id': 'bmgb0ya6m', 'function': {'arguments': '{"body":"Let\'s meet tomorrow at 

#approval
from langgraph.types import Command
if "__interrupt__" in response:
    
    response=agent.invoke(
        Command(
            resume={
                [
                    "decision": {"type":"approve"}
                ]
                         
            }
        ),
        config=config
    )
    
    print(reponse["messages"][-1]["content"])